In [1]:
from google.cloud import bigquery
import pandas as pd
from typing import Union
import re
from bokeh.io import output_notebook
from bokeh.models import ColumnDataSource, DataTable, TableColumn, CustomJS, Select, Button, Div
from bokeh.plotting import show, output_file, save
from bokeh.layouts import column, row

In [2]:
output_notebook()

Loading BokehJS ...

In [3]:
client = bigquery.Client(project='subugoe-collaborative')

In [4]:
oal_inst_lower_saxony_raw = client.query(f"""
                                          SELECT DISTINCT
                                          CASE 
                                            WHEN oal.id IS NOT NULL THEN oal.id
                                            ELSE kb.id
                                          END AS id,
                                          kb.source AS kb_source, 
                                          oal.source AS oal_source,
                                          kb.inst_name AS kb_name,
                                          oal.inst_name AS oal_name,
                                          kb.ror_id,
                                          CASE 
                                            WHEN oal.publication_year IS NOT NULL THEN oal.publication_year
                                            ELSE kb.publication_year
                                          END AS publication_year,
                                          CASE 
                                            WHEN oal.raw_affiliation_string IS NOT NULL THEN oal.raw_affiliation_string
                                            ELSE address_full
                                          END AS raw_affiliation_string
                                        FROM (
                                          SELECT o.id, 
                                                 CASE 
                                                   WHEN kb_inst.ror = 'https://ror.org/021ft0n22' THEN 'Universitätsmedizin Göttingen'
                                                   WHEN kb_inst.ror = 'https://ror.org/02w2y2t16' THEN 'Leuphana Universität Lüneburg'
                                                   WHEN kb_inst.ror = 'https://ror.org/033n9gh91' THEN 'Carl von Ossietzky Universität Oldenburg'
                                                   WHEN kb_inst.ror = 'https://ror.org/02vvvm705' THEN 'Jade Hochschule Wilhelmshaven/Oldenburg/Elsfleth'
                                                   WHEN kb_inst.ror = 'https://ror.org/01bc76c69' THEN 'Hochschule Emden/Leer'
                                                   WHEN kb_inst.ror = 'https://ror.org/0304hq317' THEN 'Gottfried Wilhelm Leibniz Universität Hannover'
                                                   WHEN kb_inst.ror = 'https://ror.org/00x67m532' THEN 'Hochschule für Musik, Theater und Medien Hannover'
                                                   WHEN kb_inst.ror = 'https://ror.org/03m2kj587' THEN 'Hochschule Hannover'
                                                   WHEN kb_inst.ror = 'https://ror.org/015qjqf64' THEN 'Stiftung Tierärztliche Hochschule Hannover'
                                                   WHEN kb_inst.ror = 'https://ror.org/00f2yqf98' THEN 'Medizinische Hochschule Hannover (MHH)'
                                                   WHEN kb_inst.ror = 'https://ror.org/00f5q5839' THEN 'HAWK Hochschule für angewandte Wissenschaft und Kunst'
                                                   WHEN kb_inst.ror = 'https://ror.org/02f9det96' THEN 'Stiftung Universität Hildesheim'
                                                   WHEN kb_inst.ror = 'https://ror.org/01y9bpm73' THEN 'Georg-August-Universität Göttingen'
                                                   WHEN kb_inst.ror = 'https://ror.org/010nsgg66' THEN 'Technische Universität Braunschweig'
                                                   WHEN kb_inst.ror = 'https://ror.org/03aft2f80' THEN 'Hochschule für Bildende Künste Braunschweig'
                                                   WHEN kb_inst.ror = 'https://ror.org/01bk10867' THEN 'Ostfalia Hochschule für angewandte Wissenschaften'
                                                   WHEN kb_inst.ror = 'https://ror.org/04qb8nc58' THEN 'Technische Universität Clausthal'
                                                   WHEN kb_inst.ror = 'https://ror.org/04qmmjx98' THEN 'Universität Osnabrück'
                                                   WHEN kb_inst.ror = 'https://ror.org/059vymd37' THEN 'Hochschule Osnabrück'
                                                   WHEN kb_inst.ror = 'https://ror.org/045y6d111' THEN 'Universität Vechta'
                                                   -- Ergänzung von Suborganisationen (Oldenburg)
                                                   WHEN inst_id = 445 THEN 'Carl von Ossietzky Universität Oldenburg' -- Evangelisches Krankenhaus Oldenburg
                                                   WHEN inst_id = 6612 THEN 'Carl von Ossietzky Universität Oldenburg' -- UMO - Universitätsmedizin Oldenburg
                                                   WHEN inst_id = 316 THEN 'Carl von Ossietzky Universität Oldenburg' -- Klinikum Oldenburg gGmbH
                                                   WHEN inst_id = 315 THEN 'Carl von Ossietzky Universität Oldenburg' -- Pius-Hospital Oldenburg
                                                   WHEN inst_id = 5488 THEN 'Carl von Ossietzky Universität Oldenburg' -- Helmholtz-Institut für Funktionelle Marine Biodiversität an der Universität Oldenburg (HIFMB)
                                                   WHEN inst_id = 4564 THEN 'Carl von Ossietzky Universität Oldenburg' -- Institute for Science Networking Oldenburg GmbH
                                                   ELSE ''
                                                 END AS inst_name,
                                                 kb_inst.ror AS ror_id,
                                                 address_full, 
                                                 REGEXP_REPLACE(address_full, r'[.,\s+]', '') AS raw_affiliation_string_cleaned,
                                                 publication_year, 
                                                 'KB' AS source
                                          FROM `subugoe-collaborative.resources.kb_a_addr_inst_202603` AS inst
                                          JOIN `subugoe-collaborative.resources.add_institution_lookup_kb_suppl_202603` AS kb_inst
                                            ON inst.inst_id_top = kb_inst.inst_id
                                          JOIN `subugoe-collaborative.openalex_walden.works` AS o
                                              ON CONCAT('https://openalex.org/', inst.item_id) = o.id
                                          WHERE o.type IN ('article', 'review') 
                                              AND primary_location.source.type = 'journal'
                                              AND is_paratext=FALSE 
                                              AND is_retracted=FALSE 
                                              AND is_xpac=FALSE
                                              AND publication_year BETWEEN 2020 AND 2024
                                              AND (kb_inst.ror IN (
                                                  'https://ror.org/021ft0n22', -- Universitätsmedizin Göttingen
                                                  'https://ror.org/02w2y2t16', -- Leuphana Universität Lüneburg
                                                  'https://ror.org/033n9gh91', -- Carl von Ossietzky Universität Oldenburg
                                                  'https://ror.org/02vvvm705', -- Jade Hochschule Wilhelmshaven/Oldenburg/Elsfleth
                                                  'https://ror.org/01bc76c69', -- Hochschule Emden/Leer
                                                  'https://ror.org/0304hq317', -- Gottfried Wilhelm Leibniz Universität Hannover
                                                  'https://ror.org/00x67m532', -- Hochschule für Musik, Theater und Medien Hannover
                                                  'https://ror.org/03m2kj587', -- Hochschule Hannover
                                                  'https://ror.org/015qjqf64', -- Stiftung Tierärztliche Hochschule Hannover
                                                  'https://ror.org/00f2yqf98', -- Medizinische Hochschule Hannover (MHH)
                                                  'https://ror.org/00f5q5839', -- HAWK Hochschule für angewandte Wissenschaft und Kunst
                                                  'https://ror.org/02f9det96', -- Stiftung Universität Hildesheim
                                                  'https://ror.org/01y9bpm73', -- Georg-August-Universität Göttingen
                                                  'https://ror.org/010nsgg66', -- Technische Universität Braunschweig
                                                  'https://ror.org/03aft2f80', -- Hochschule für Bildende Künste Braunschweig
                                                  'https://ror.org/01bk10867', -- Ostfalia Hochschule für angewandte Wissenschaften
                                                  'https://ror.org/04qb8nc58', -- Technische Universität Clausthal
                                                  'https://ror.org/04qmmjx98', -- Universität Osnabrück
                                                  'https://ror.org/059vymd37', -- Hochschule Osnabrück
                                                  'https://ror.org/045y6d111' -- Universität Vechta
                                              ) OR kb_inst.inst_id IN (
                                                    -- Ergänzung von Suborganisationen (Oldenburg)
                                                    445, -- 'Evangelisches Krankenhaus Oldenburg'
                                                    6612, -- 'UMO - Universitätsmedizin Oldenburg'
                                                    316, -- 'Klinikum Oldenburg gGmbH'
                                                    315, -- 'Pius-Hospital Oldenburg'
                                                    5488, -- 'Helmholtz-Institut für Funktionelle Marine Biodiversität an der Universität Oldenburg (HIFMB)'
                                                    4564 -- Institute for Science Networking Oldenburg GmbH
                                                )
                                              )
                                        ) AS kb
                                        FULL OUTER JOIN (
                                            SELECT oal.id, 
                                               CASE 
                                                 WHEN inst.ror = 'https://ror.org/02w2y2t16' THEN 'Leuphana Universität Lüneburg'
                                                 WHEN inst.ror = 'https://ror.org/033n9gh91' THEN 'Carl von Ossietzky Universität Oldenburg'
                                                 WHEN inst.ror = 'https://ror.org/02vvvm705' THEN 'Jade Hochschule Wilhelmshaven/Oldenburg/Elsfleth'
                                                 WHEN inst.ror = 'https://ror.org/01bc76c69' THEN 'Hochschule Emden/Leer'
                                                 WHEN inst.ror = 'https://ror.org/0304hq317' THEN 'Gottfried Wilhelm Leibniz Universität Hannover'
                                                 WHEN inst.ror = 'https://ror.org/00x67m532' THEN 'Musikhochschule Hannover'
                                                 WHEN inst.ror = 'https://ror.org/03m2kj587' THEN 'Hochschule für Musik, Theater und Medien Hannover'
                                                 WHEN inst.ror = 'https://ror.org/015qjqf64' THEN 'Stiftung Tierärztliche Hochschule Hannover'
                                                 WHEN inst.ror = 'https://ror.org/00f2yqf98' THEN 'Medizinische Hochschule Hannover (MHH)'
                                                 WHEN inst.ror = 'https://ror.org/00f5q5839' THEN 'HAWK Hochschule für angewandte Wissenschaft und Kunst'
                                                 WHEN inst.ror = 'https://ror.org/02f9det96' THEN 'Stiftung Universität Hildesheim'
                                                 WHEN inst.ror = 'https://ror.org/01y9bpm73' THEN 'Georg-August-Universität Göttingen'
                                                 WHEN inst.ror = 'https://ror.org/010nsgg66' THEN 'Technische Universität Braunschweig'
                                                 WHEN inst.ror = 'https://ror.org/03aft2f80' THEN 'Hochschule für Bildende Künste Braunschweig'
                                                 WHEN inst.ror = 'https://ror.org/01bk10867' THEN 'Ostfalia Hochschule für angewandte Wissenschaften'
                                                 WHEN inst.ror = 'https://ror.org/04qb8nc58' THEN 'Technische Universität Clausthal'
                                                 WHEN inst.ror = 'https://ror.org/04qmmjx98' THEN 'Universität Osnabrück'
                                                 WHEN inst.ror = 'https://ror.org/059vymd37' THEN 'Hochschule Osnabrück'
                                                 WHEN inst.ror = 'https://ror.org/045y6d111' THEN 'Universität Vechta'
                                                 -- Ergänzung von Suborganisationen (GAU)
                                                 WHEN inst.ror = 'https://ror.org/021ft0n22' THEN 'Georg-August-Universität Göttingen' -- UMG in GAU integrieren für Vergleichbarkeit
                                                 WHEN inst.ror = 'https://ror.org/00cd95c65' THEN 'Georg-August-Universität Göttingen' -- Gesellschaft für wissenschaftliche Datenverarbeitung mbH Göttingen
                                                 WHEN inst.ror = 'https://ror.org/02f04tm31' THEN 'Georg-August-Universität Göttingen' -- Göttingen Campus Institut für Dynamic biologischer Netzwerke
                                                 WHEN inst.ror = 'https://ror.org/044sxzm68' THEN 'Georg-August-Universität Göttingen' -- Campus-Institut Data Science (CIDAS)
                                                 WHEN inst.ror = 'https://ror.org/05745n787' THEN 'Georg-August-Universität Göttingen' -- Niedersächsische Staats-und Universitätsbibliothek Göttingen
                                                 WHEN inst.ror = 'https://ror.org/03vwt8p73' THEN 'Georg-August-Universität Göttingen' -- Else Kröner Fresenius Zentrum für Optogenetische Therapien
                                                 -- Ergänzung von Suborganisationen (LUH)
                                                 WHEN inst.ror = 'https://ror.org/039t4wk02' THEN 'Gottfried Wilhelm Leibniz Universität Hannover' -- Forschungszentrum L3S
                                                 WHEN inst.ror = 'https://ror.org/00w53fs94' THEN 'Gottfried Wilhelm Leibniz Universität Hannover' -- Forschungszentrum Küste (FZK)
                                                 -- Ergänzung von Suborganisationen (Oldenburg)
                                                 WHEN inst.ror = 'https://ror.org/025t8vx68' THEN 'Carl von Ossietzky Universität Oldenburg' -- Institut für Ökonomische Bildung
                                                 WHEN inst.ror = 'https://ror.org/0060pja03' THEN 'Carl von Ossietzky Universität Oldenburg' -- Institut für Chemie und Biologie des Meeres
                                                 WHEN inst.ror = 'https://ror.org/01t0n2c80' THEN 'Carl von Ossietzky Universität Oldenburg' -- Klinikum Oldenburg
                                                 WHEN inst.ror = 'https://ror.org/04830hf15' THEN 'Carl von Ossietzky Universität Oldenburg' -- Evangelisches Krankenhaus Oldenburg
                                                 WHEN inst.ror = 'https://ror.org/03avbdx23' THEN 'Carl von Ossietzky Universität Oldenburg' -- Pius Hospital Oldenburg
                                                 WHEN inst.ror = 'https://ror.org/00tea5y39' THEN 'Carl von Ossietzky Universität Oldenburg' -- Helmholtz-Institut für Funktionelle Marine Biodiversität
                                                 ELSE ''
                                               END AS inst_name,
                                               inst.ror AS oal_id,
                                               raw_affiliation_string,
                                               REGEXP_REPLACE(raw_affiliation_string, r'[.,\s+]', '') AS raw_affiliation_string_cleaned,
                                               publication_year, 
                                               'OAL' AS source
                                            FROM `subugoe-collaborative.openalex_walden.works` AS oal
                                            LEFT JOIN UNNEST(authorships) AS aut
                                            LEFT JOIN UNNEST(aut.affiliations) AS aff
                                            JOIN `subugoe-collaborative.openalex_walden.institutions` AS inst
                                                ON aff.institution_ids[SAFE_OFFSET(0)] = inst.id
                                            WHERE oal.type IN ('article', 'review') 
                                                AND primary_location.source.type = 'journal'
                                                AND is_paratext=FALSE 
                                                AND is_retracted=FALSE 
                                                AND is_xpac=FALSE
                                                AND publication_year BETWEEN 2020 AND 2024
                                                AND inst.ror IN (
                                                    'https://ror.org/021ft0n22', -- Universitätsmedizin Göttingen
                                                    'https://ror.org/02w2y2t16', -- Leuphana Universität Lüneburg
                                                    'https://ror.org/033n9gh91', -- Carl von Ossietzky Universität Oldenburg
                                                    'https://ror.org/02vvvm705', -- Jade Hochschule Wilhelmshaven/Oldenburg/Elsfleth
                                                    'https://ror.org/01bc76c69', -- Hochschule Emden/Leer
                                                    'https://ror.org/0304hq317', -- Gottfried Wilhelm Leibniz Universität Hannover
                                                    'https://ror.org/00x67m532', -- Hochschule für Musik, Theater und Medien Hannover
                                                    'https://ror.org/03m2kj587', -- Hochschule Hannover
                                                    'https://ror.org/015qjqf64', -- Stiftung Tierärztliche Hochschule Hannover
                                                    'https://ror.org/00f2yqf98', -- Medizinische Hochschule Hannover (MHH)
                                                    'https://ror.org/00f5q5839', -- HAWK Hochschule für angewandte Wissenschaft und Kunst
                                                    'https://ror.org/02f9det96', -- Stiftung Universität Hildesheim
                                                    'https://ror.org/01y9bpm73', -- Georg-August-Universität Göttingen
                                                    'https://ror.org/010nsgg66', -- Technische Universität Braunschweig
                                                    'https://ror.org/03aft2f80', -- Hochschule für Bildende Künste Braunschweig
                                                    'https://ror.org/01bk10867', -- Ostfalia Hochschule für angewandte Wissenschaften
                                                    'https://ror.org/04qb8nc58', -- Technische Universität Clausthal
                                                    'https://ror.org/04qmmjx98', -- Universität Osnabrück
                                                    'https://ror.org/059vymd37', -- Hochschule Osnabrück
                                                    'https://ror.org/045y6d111', -- Universität Vechta
                                                    -- Ergänzung von Suborganisationen (GAU)
                                                    'https://ror.org/00cd95c65', -- Gesellschaft für wissenschaftliche Datenverarbeitung mbH Göttingen
                                                    'https://ror.org/02f04tm31', -- Göttingen Campus Institut für Dynamic biologischer Netzwerke
                                                    'https://ror.org/044sxzm68', -- Campus-Institut Data Science (CIDAS)
                                                    'https://ror.org/05745n787', -- Niedersächsische Staats-und Universitätsbibliothek Göttingen
                                                    'https://ror.org/03vwt8p73', -- Else Kröner Fresenius Zentrum für Optogenetische Therapien
                                                    -- Ergänzung von Suborganisationen (LUH)
                                                    'https://ror.org/039t4wk02', -- Forschungszentrum L3S
                                                    'https://ror.org/00w53fs94', -- Forschungszentrum Küste (FZK)
                                                    -- Ergänzung von Suborganisationen (Oldenburg)
                                                    'https://ror.org/025t8vx68', -- Institut für Ökonomische Bildung
                                                    'https://ror.org/0060pja03', -- Institut für Chemie und Biologie des Meeres
                                                    'https://ror.org/01t0n2c80', -- Klinikum Oldenburg
                                                    'https://ror.org/04830hf15' -- Evangelisches Krankenhaus Oldenburg
                                                    'https://ror.org/03avbdx23' -- Pius Hospital Oldenburg
                                                    'https://ror.org/00tea5y39' -- Helmholtz-Institut für Funktionelle Marine Biodiversität
                                                )
                                            ) AS oal
                                        ON kb.id = oal.id
                                        AND LOWER(kb.raw_affiliation_string_cleaned) = LOWER(oal.raw_affiliation_string_cleaned)
                            """).to_dataframe()

In [6]:
#oal_inst_lower_saxony_raw.to_csv('../data/inst_list_full_with_aff_strings_cleaned.csv', sep=',', index=False)

In [7]:
oal_inst_lower_saxony = pd.read_csv('../data/inst_list_full_with_aff_strings_cleaned.csv')

In [8]:
oal_inst_lower_saxony.head()

,id,kb_source,oal_source,kb_name,oal_name,ror_id,publication_year,raw_affiliation_string
0,https://openalex.org/W4403424220,KB,OAL,Ostfalia Hochschule für angewandte Wissenschaften,Ostfalia Hochschule für angewandte Wissenschaften,https://ror.org/01bk10867,2024,"Ostfalia University of Applied Sciences, Salzd..."
1,https://openalex.org/W4388731974,KB,OAL,Jade Hochschule Wilhelmshaven/Oldenburg/Elsfleth,Jade Hochschule Wilhelmshaven/Oldenburg/Elsfleth,https://ror.org/02vvvm705,2023,"Department of Engineering, Jade University of ..."
2,https://openalex.org/W4318763113,KB,NaN,HAWK Hochschule für angewandte Wissenschaft un...,NaN,https://ror.org/00f5q5839,2023,"Faculty of Management, Social Work, and Constr..."
3,https://openalex.org/W4282973240,KB,NaN,Hochschule Hannover,NaN,https://ror.org/03m2kj587,2022,"Department of Information and Communication, F..."
4,https://openalex.org/W4307382744,KB,OAL,Hochschule Emden/Leer,Hochschule Emden/Leer,https://ror.org/01bc76c69,2022,"University of Applied Sciences Emden/Leer, Bra..."


In [9]:
oal_inst_lower_saxony[oal_inst_lower_saxony.id == 'https://openalex.org/W3012291016']

,id,kb_source,oal_source,kb_name,oal_name,ror_id,publication_year,raw_affiliation_string
17759,https://openalex.org/W3012291016,KB,OAL,Medizinische Hochschule Hannover (MHH),Medizinische Hochschule Hannover (MHH),https://ror.org/00f2yqf98,2020,Klinik für Strahlentherapie und Spezielle Onko...


In [10]:
oal_inst_lower_saxony[oal_inst_lower_saxony.id == 'https://openalex.org/W112007689']

,id,kb_source,oal_source,kb_name,oal_name,ror_id,publication_year,raw_affiliation_string
3905,https://openalex.org/W112007689,NaN,OAL,NaN,Georg-August-Universität Göttingen,NaN,2024,Götting KG
88335,https://openalex.org/W112007689,KB,OAL,Gottfried Wilhelm Leibniz Universität Hannover,Gottfried Wilhelm Leibniz Universität Hannover,https://ror.org/0304hq317,2024,Institut für Transport- und Automatisierungste...


In [11]:
def make_set(list_of_names: list): 
    return set(name for name in list_of_names if pd.notna(name))

In [12]:
kb_list = oal_inst_lower_saxony.groupby(['id', 'raw_affiliation_string', 'publication_year'])['kb_name'].apply(make_set).reset_index()
oal_list = oal_inst_lower_saxony.groupby(['id', 'raw_affiliation_string', 'publication_year'])['oal_name'].apply(make_set).reset_index()
inst_list = pd.merge(kb_list, oal_list, on=['id', 'raw_affiliation_string', 'publication_year'], how='outer')

In [13]:
inst_list['kb_name'] = inst_list['kb_name'].fillna('').apply(make_set)
inst_list['oal_name'] = inst_list['oal_name'].fillna('').apply(make_set)

inst_list['in_oal_missing'] = list(inst_list['kb_name'] - inst_list['oal_name'])
inst_list['in_kb_missing'] = list(inst_list['oal_name'] - inst_list['kb_name'])

inst_list['kb_count'] = inst_list.kb_name.str.len()
inst_list['oal_count'] = inst_list.oal_name.str.len()

In [14]:
inst_list.head()

,id,raw_affiliation_string,publication_year,kb_name,oal_name,in_oal_missing,in_kb_missing,kb_count,oal_count
0,https://openalex.org/W112007689,Götting KG,2024,{},{Georg-August-Universität Göttingen},{},{Georg-August-Universität Göttingen},0,1
1,https://openalex.org/W112007689,Institut für Transport- und Automatisierungste...,2024,{Gottfried Wilhelm Leibniz Universität Hannover},{Gottfried Wilhelm Leibniz Universität Hannover},{},{},1,1
2,https://openalex.org/W1483587807,"Centre Georg Simmel, Recherches franco-alleman...",2021,{Leuphana Universität Lüneburg},{},{Leuphana Universität Lüneburg},{},1,0
3,https://openalex.org/W1483587807,Leuphana University Lueneburg,2021,{Leuphana Universität Lüneburg},{Leuphana Universität Lüneburg},{},{},1,1
4,https://openalex.org/W1500095539,"University and Polytechnic of Lüneburg, German...",2024,{},{Leuphana Universität Lüneburg},{},{Leuphana Universität Lüneburg},0,1


In [15]:
df = inst_list[['id', 'raw_affiliation_string', 'publication_year', 'in_oal_missing', 'in_kb_missing']].copy()

In [16]:
df = df.assign(in_oal_missing=df['in_oal_missing']).explode('in_oal_missing').reset_index(drop=True)
df = df.assign(in_kb_missing=df['in_kb_missing']).explode('in_kb_missing').reset_index(drop=True)

In [17]:
df

,id,raw_affiliation_string,publication_year,in_oal_missing,in_kb_missing
0,https://openalex.org/W112007689,Götting KG,2024,NaN,Georg-August-Universität Göttingen
1,https://openalex.org/W112007689,Institut für Transport- und Automatisierungste...,2024,NaN,NaN
2,https://openalex.org/W1483587807,"Centre Georg Simmel, Recherches franco-alleman...",2021,Leuphana Universität Lüneburg,NaN
3,https://openalex.org/W1483587807,Leuphana University Lueneburg,2021,NaN,NaN
4,https://openalex.org/W1500095539,"University and Polytechnic of Lüneburg, German...",2024,NaN,Leuphana Universität Lüneburg
...,...,...,...,...,...
114475,https://openalex.org/W7155016106,"University of Vechta, Faculty II, Geography & ...",2024,NaN,Universität Vechta
114476,https://openalex.org/W7160830967,"University of Goettingen, Germany",2024,NaN,Georg-August-Universität Göttingen
114477,https://openalex.org/W7161541681,former docente at the Institute of Music at th...,2024,NaN,Carl von Ossietzky Universität Oldenburg
114478,https://openalex.org/W7163685763,"Fachbereich BGG, Jade Hochschule Oldenburg, Ge...",2024,NaN,Jade Hochschule Wilhelmshaven/Oldenburg/Elsfleth


In [18]:
df.dropna(subset=['in_oal_missing', 'in_kb_missing'], how='all', inplace=True)

In [19]:
df = df[~df.publication_year.isnull()]
df['publication_year'] = df['publication_year'].astype(int)

In [20]:
df.head()

,id,raw_affiliation_string,publication_year,in_oal_missing,in_kb_missing
0,https://openalex.org/W112007689,Götting KG,2024,NaN,Georg-August-Universität Göttingen
2,https://openalex.org/W1483587807,"Centre Georg Simmel, Recherches franco-alleman...",2021,Leuphana Universität Lüneburg,NaN
4,https://openalex.org/W1500095539,"University and Polytechnic of Lüneburg, German...",2024,NaN,Leuphana Universität Lüneburg
11,https://openalex.org/W173619620,Universität Lüneburg Zentrum für Angewandte Ge...,2024,Leuphana Universität Lüneburg,NaN
13,https://openalex.org/W17584880,"t, Braunschweig,",2021,NaN,Technische Universität Braunschweig


In [21]:
df[(~df.in_kb_missing.isnull()) & (df.in_kb_missing != 'nan')]

,id,raw_affiliation_string,publication_year,in_oal_missing,in_kb_missing
0,https://openalex.org/W112007689,Götting KG,2024,NaN,Georg-August-Universität Göttingen
4,https://openalex.org/W1500095539,"University and Polytechnic of Lüneburg, German...",2024,NaN,Leuphana Universität Lüneburg
13,https://openalex.org/W17584880,"t, Braunschweig,",2021,NaN,Technische Universität Braunschweig
21,https://openalex.org/W1919750199,Heidelberg University University of Göttingen ...,2020,NaN,Georg-August-Universität Göttingen
28,https://openalex.org/W2016255857,Götting KG,2024,NaN,Georg-August-Universität Göttingen
...,...,...,...,...,...
114474,https://openalex.org/W7155016106,"University of Vechta, Faculty II, Biology & Ve...",2024,NaN,Universität Vechta
114475,https://openalex.org/W7155016106,"University of Vechta, Faculty II, Geography & ...",2024,NaN,Universität Vechta
114476,https://openalex.org/W7160830967,"University of Goettingen, Germany",2024,NaN,Georg-August-Universität Göttingen
114477,https://openalex.org/W7161541681,former docente at the Institute of Music at th...,2024,NaN,Carl von Ossietzky Universität Oldenburg


In [22]:
df[df.id == 'https://openalex.org/W3030680563']

,id,raw_affiliation_string,publication_year,in_oal_missing,in_kb_missing
8810,https://openalex.org/W3030680563,"Klinik für Orthopädie im Annastift, Medizinisc...",2020,NaN,Medizinische Hochschule Hannover (MHH)


In [23]:
#df[(~df.in_kb_missing.isnull()) & (df.in_kb_missing != 'nan')].to_csv('missing_inst_in_kb.csv', index=False)

In [24]:
true_df = df.copy()
true_df['in_kb_missing'] = true_df['in_kb_missing'].replace('nan', '')
true_df['in_oal_missing'] = true_df['in_oal_missing'].replace('nan', '')

In [25]:
true_df.id.count()

18114

In [26]:
# Universitätsmedizin Göttingen
# https://ror.org/021ft0n22 
umg = ['univ(ersi(t)?((ä|a|ae)ts|y)|.)?(\s|\sof\s)?(med|hosp|klinik)', 'gyn(äk|ec)olog(ie|y)', 'dentistry', 'cardiovascular',
       'medical (university )?(cent(re|er)|school|clinic|science)?', 'neuroimmunology', 'cardiology', 'psychosomatic',
       'h((a)?e|ä)matolog(ie|y)', 'radiolog(ie|y)', 'neurolog(ie|y)', 'medical (bio)?informatics', 'pneumology', '(augen|kinder|frauen)klinik',
       'pharmacology', 'an((a)?e|ä)sthesiolog(ie|y)', 'bioimaging', 'psychiatry', 'psychoneuroscience', 'dental', 'anatomy',
       'heart cent(er|re)', '(palliativ|intensiv)medizin', 'cellular biophysics', 'medizinische statistik', 'nephrolog(ie|y)',
       'infektiologie', '(trauma|neuro|thoracic|neck)(\s)?surgery', 'epidemiology', 'medizin und psychotherapie',
       'otorhinolaryngology', 'maxillofacial', 'jugendpsychiatrie', 'molecular cell', 'gastroenterolog(ie|y)',
       'otolaryngology', '889', '1690', '1286', '(department|faculty) of medicine', 'dermatolog(ie|y)', 'infectious diseases',
       'nuclear medicine', 'orthop(a)?edics', '(gesichts|unfall|kinder)chirurgie', 'hno(\s|-)((uni)?klinik|göttingen)',
       'cellular biochemistry', 'forensisch', 'geriatri(e|cs)', 'frauenheilkunde', "children's hospital",
       'klinik für psychiatrie', 'neuropathology', 'p(a)?ed(r)?iat(r)?ic', 'clinical psychology', 'ophthalmol',
       'cardiovasc', 'molecular oncology', 'virology', 'sensory physiology', 'general (practice|surgery)',
       'human(\s)?geneti(k|cs)', 'rheumatolog(ie|y)', 'institute of pathology', 'radiotherapy', 'neurophysiolog(ie|y)',
       'medizinische fakultät', '(umg|uniklinik) göttingen', 'molekulare psychiatrie', 'synaptic physiology',
       'integrative genomics']

In [27]:
def map_umg(address: str) -> Union[str, None]:
    pattern = re.compile(r'|'.join(umg), re.IGNORECASE)
    res = bool(pattern.search(address))
    if res:
        return  'Universitätsmedizin Göttingen'
    else:
        return 'Georg-August-Universität Göttingen'

In [28]:
true_df.loc[true_df['in_oal_missing'] == 'Georg-August-Universität Göttingen', 'in_oal_missing'] = \
           true_df.loc[true_df['in_oal_missing'] == 'Georg-August-Universität Göttingen']['raw_affiliation_string'].apply(map_umg)

true_df.loc[true_df['in_kb_missing'] == 'Georg-August-Universität Göttingen', 'in_kb_missing'] = \
           true_df.loc[true_df['in_kb_missing'] == 'Georg-August-Universität Göttingen']['raw_affiliation_string'].apply(map_umg)

In [29]:
true_df

,id,raw_affiliation_string,publication_year,in_oal_missing,in_kb_missing
0,https://openalex.org/W112007689,Götting KG,2024,NaN,Georg-August-Universität Göttingen
2,https://openalex.org/W1483587807,"Centre Georg Simmel, Recherches franco-alleman...",2021,Leuphana Universität Lüneburg,NaN
4,https://openalex.org/W1500095539,"University and Polytechnic of Lüneburg, German...",2024,NaN,Leuphana Universität Lüneburg
11,https://openalex.org/W173619620,Universität Lüneburg Zentrum für Angewandte Ge...,2024,Leuphana Universität Lüneburg,NaN
13,https://openalex.org/W17584880,"t, Braunschweig,",2021,NaN,Technische Universität Braunschweig
...,...,...,...,...,...
114474,https://openalex.org/W7155016106,"University of Vechta, Faculty II, Biology & Ve...",2024,NaN,Universität Vechta
114475,https://openalex.org/W7155016106,"University of Vechta, Faculty II, Geography & ...",2024,NaN,Universität Vechta
114476,https://openalex.org/W7160830967,"University of Goettingen, Germany",2024,NaN,Georg-August-Universität Göttingen
114477,https://openalex.org/W7161541681,former docente at the Institute of Music at th...,2024,NaN,Carl von Ossietzky Universität Oldenburg


In [30]:
source = ColumnDataSource(data=true_df)

In [31]:
all_inst = sorted(set(name for name in true_df.in_kb_missing.tolist() if pd.notna(name)) | set(name for name in true_df.in_oal_missing.tolist() if pd.notna(name)))

In [37]:
output_file(filename='download.html', title='HDN-FIS: Institutionen Download')

columns = [
    TableColumn(field='id', title='OpenAlex ID', width=200),
    TableColumn(field='publication_year', title='Publikationsjahr', width=100),
    TableColumn(field='raw_affiliation_string', width=700, title='Affiliationsstring'),
    TableColumn(field='in_oal_missing', title='Institutionszuordnung durch KB'),
    TableColumn(field='in_kb_missing', title='Institutionszuordnung durch OpenAlex')
]

data_table = DataTable(source=source, 
                       columns=columns, 
                       editable=True, 
                       selectable=True,
                       row_height=35,
                       width=1600,
                       height=500
                      )

source.data = dict(true_df[(true_df.in_oal_missing == 'Carl von Ossietzky Universität Oldenburg') | (true_df.in_kb_missing == 'Carl von Ossietzky Universität Oldenburg')])

select_inst = Select(title='Institution', 
                     width=400,
                     value='Carl von Ossietzky Universität Oldenburg', 
                     options=all_inst)

select_year = Select(title='Publikationsjahr', 
                     width=200,
                     value='Alle', 
                     options=['Alle'] + [str(year) for year in sorted(list(true_df.publication_year.unique()))])

div_row_count = Div(text=f"Treffer: <b>{len(source.data.get('id'))}</b>")

callback = CustomJS(args=dict(source=source, 
                              row_div=div_row_count,
                              year=select_year, 
                              inst=select_inst, 
                              original_data=true_df.to_dict('list')), 
                    code="""
    let filtered_data = {
        'index': [],
        'id': [],
        'publication_year': [],
        'raw_affiliation_string': [],
        'in_oal_missing': [],
        'in_kb_missing': []
    };

    console.log(original_data);

    if (year.value == 'Alle') {
        for (let i = 0; i < original_data['id'].length; i++) {
            if (original_data['in_oal_missing'][i] == inst.value || original_data['in_kb_missing'][i] == inst.value) {
                filtered_data['index'].push(original_data[i]);
                filtered_data['id'].push(original_data['id'][i]);
                filtered_data['publication_year'].push(original_data['publication_year'][i]);
                filtered_data['raw_affiliation_string'].push(original_data['raw_affiliation_string'][i]);
                filtered_data['in_oal_missing'].push(original_data['in_oal_missing'][i]);
                filtered_data['in_kb_missing'].push(original_data['in_kb_missing'][i]);
            }
        }
    } else {

        for (let i = 0; i < original_data['id'].length; i++) {
            if (original_data['publication_year'][i] == parseInt(year.value) && (original_data['in_oal_missing'][i] == inst.value || original_data['in_kb_missing'][i] == inst.value)) {
                filtered_data['index'].push(original_data[i]);
                filtered_data['id'].push(original_data['id'][i]);
                filtered_data['publication_year'].push(original_data['publication_year'][i]);
                filtered_data['raw_affiliation_string'].push(original_data['raw_affiliation_string'][i]);
                filtered_data['in_oal_missing'].push(original_data['in_oal_missing'][i]);
                filtered_data['in_kb_missing'].push(original_data['in_kb_missing'][i]);
            }
        }
    }

    const row_count = filtered_data['id'].length;

    source.data = filtered_data;
    row_div.text = `Treffer: <b>${row_count}</b>`;
""")

callback_download = CustomJS(args=dict(source=source),
                             code="""
        // FROM: https://github.com/bokeh/bokeh/blob/main/examples/server/app/export_csv/download.js
        let csv = 'OpenAlex ID;Publikationsjahr;Affiliationsstring;Fehlende Institutionen in OpenAlex\\n';
        for (let i = 0; i < source.data['id'].length; i++) {
            csv += source.data['id'][i] + ';' + source.data['publication_year'][i] + ';' + source.data['raw_affiliation_string'][i] + ';' + source.data['in_oal_missing'][i] + '\\n';
        }     

        const filename = 'FehlendeInstitutionen.csv'
        const blob = new Blob([csv], {type: 'text/csv;charset=utf-8;'})
        
        //addresses IE
        if (navigator.msSaveBlob) {
            navigator.msSaveBlob(blob, filename)
        } else {
            const link = document.createElement('a')
            link.href = URL.createObjectURL(blob)
            link.download = filename
            link.target = '_blank'
            link.style.visibility = 'hidden'
            link.dispatchEvent(new MouseEvent('click'))
        }
""")

select_inst.js_on_change('value', callback)
select_year.js_on_change('value', callback)

button = Button(label='Download', button_type='default', margin=(22, 0, 0, 15))
button.js_on_event('button_click', callback_download)

div_header = Div(text="""
                      <h1 style=font-size:16px>HDN-FIS: Institutionsanreicherung niedersächsicher Hochschulen in OpenAlex - Download</h1>
                      """
                )

div_text = Div(text="""
                    <p>Die folgende Tabelle enthält Publikationen, bei denen die Institutionszuordnung im KB unterschiedlich zu der in OpenAlex ist. <br>
                    Es werden nur Publikationen angezeigt, die eine Zuordnung zu einer niedersächsischen Hochschule haben und zwischen 2020 und 2024 erschienen sind. </p>
                    <br>
                    <p><b>Datenquellen:</b></p>
                    <ul>
                        <li><b>OpenAlex:</b> Stand Juni 2026</li>
                        <li><b>KB:</b> Stand März 2026</li>
                    </ul>
                    """
                )

#layout = column(row(select_inst, select_year, button, spacing=50), data_table, spacing=15)
layout = column(
            column(div_header, div_text), 
            row(select_inst, select_year, button, spacing=50), 
            div_row_count, 
            data_table, 
            spacing=15
)

show(layout)
save(layout)

'/Users/naustica/Desktop/lower_saxony_institutions/notebooks/download.html'

In [33]:
oal_inst_lower_saxony[oal_inst_lower_saxony.id == 'https://openalex.org/W4378714479']

,id,kb_source,oal_source,kb_name,oal_name,ror_id,publication_year,raw_affiliation_string
84782,https://openalex.org/W4378714479,KB,OAL,Leuphana Universität Lüneburg,Leuphana Universität Lüneburg,https://ror.org/02w2y2t16,2023,"Institute of Political Science, Leuphana Unive..."
85760,https://openalex.org/W4378714479,KB,OAL,Leuphana Universität Lüneburg,Leuphana Universität Lüneburg,https://ror.org/02w2y2t16,2023,Institute of Political Science Leuphana Univer...


In [34]:
oal_inst_lower_saxony[oal_inst_lower_saxony.id == 'https://openalex.org/W4378714479']

,id,kb_source,oal_source,kb_name,oal_name,ror_id,publication_year,raw_affiliation_string
84782,https://openalex.org/W4378714479,KB,OAL,Leuphana Universität Lüneburg,Leuphana Universität Lüneburg,https://ror.org/02w2y2t16,2023,"Institute of Political Science, Leuphana Unive..."
85760,https://openalex.org/W4378714479,KB,OAL,Leuphana Universität Lüneburg,Leuphana Universität Lüneburg,https://ror.org/02w2y2t16,2023,Institute of Political Science Leuphana Univer...


In [35]:
oal_inst_lower_saxony[oal_inst_lower_saxony.id == 'https://openalex.org/W3002602652'].raw_affiliation_string.tolist()

['Department of Physics, Osnabrueck University, 49069 Osnabrueck, Germany. Electronic address: hsteinho@uni-osnabrueck.de',
 'Department of Physics, Osnabrueck University, 49069 Osnabrueck, Germany']

In [36]:
oal_inst_lower_saxony[oal_inst_lower_saxony.id == 'https://openalex.org/W7160830967']

,id,kb_source,oal_source,kb_name,oal_name,ror_id,publication_year,raw_affiliation_string
5279,https://openalex.org/W7160830967,NaN,OAL,NaN,Georg-August-Universität Göttingen,NaN,2024,"University of Goettingen, Germany"
